# Tracker improvement for presentation

Consider the scenario where the target evolves according to the Langevin model, driven by a normal sigma-mean mixture with the mixing distribution being the $\alpha$-stable distribution.

In [ ]:
import numpy as np
from datetime import datetime, timedelta

The state of the target can be represented as 2D Cartesian coordinates, $\left[x, \dot x, y, \dot y\right]^{\top}$, modelling both its position and velocity. A simple truth path is created with a sampling rate of 1 Hz.

In [ ]:
from stonesoup.types.groundtruth import GroundTruthPath, GroundTruthState
from stonesoup.models.base_driver import NoiseCase
from stonesoup.models.driver import AlphaStableNSMDriver 
from stonesoup.models.transition.levy_linear import LevyLangevin, CombinedLinearLevyTransitionModel

# And the clock starts
start_time = datetime.now().replace(microsecond=0)

In [ ]:
seed = 3 # Random seem for reproducibility

# Driving process parameters
mu_W = 0.02
sigma_W2 = 4
alpha = 1.4
c=10
noise_case= NoiseCase.GAUSSIAN_APPROX

# Model parameters
theta=0.15

driver_x = AlphaStableNSMDriver(mu_W=mu_W, sigma_W2=sigma_W2, seed=seed, c=c, alpha=alpha, noise_case=noise_case)
driver_y = driver_x # Same driving process in both dimensions and sharing the same latents (jumps)
langevin_x = LevyLangevin(driver=driver_x, damping_coeff=theta, mu_W=mu_W)
langevin_y = LevyLangevin(driver=driver_y, damping_coeff=theta)
transition_model = CombinedLinearLevyTransitionModel([langevin_x, langevin_y])

The ground truth is initialised from (0,0).

In [ ]:
num_steps = 40
number_particles = 100

In [ ]:
timesteps = [start_time]

truth = GroundTruthPath([GroundTruthState([0, 1, 0, 1], timestamp=timesteps[0])])



for k in range(num_steps):
    timesteps.append(start_time+timedelta(seconds=1*(k+1)))  # add next timestep to list of timesteps
    truth.append(GroundTruthState(
        transition_model.function(truth[k], noise=True, time_interval=timedelta(seconds=1)),
        timestamp=timesteps[k+1]))


from stonesoup.types.detection import Detection
from stonesoup.models.measurement.linear import LinearGaussian

measurement_model = LinearGaussian(
    ndim_state=4,  # Number of state dimensions (position and velocity in 2D)
    mapping=(0, 2),  # Mapping measurement vector index to state index
    noise_covar=np.array([[1500, 0],  # Covariance matrix for Gaussian PDF
                          [0, 1500]])
    )

measurements = []
for state in truth:
    measurement = measurement_model.function(state, noise=True)
    measurements.append(Detection(measurement,
                                  timestamp=state.timestamp,
                                  measurement_model=measurement_model))
    

## Marginalised Particle Filtering

In [ ]:
from stonesoup.predictor.particle import MarginalisedParticlePredictor
from stonesoup.resampler.particle import SystematicResampler 
from stonesoup.updater.particle import MarginalisedParticleUpdater

predictor = MarginalisedParticlePredictor(transition_model=transition_model)
resampler = SystematicResampler()
updater = MarginalisedParticleUpdater(measurement_model, resampler)

from scipy.stats import multivariate_normal
from stonesoup.types.numeric import Probability  # Similar to a float type
from stonesoup.types.state import MarginalisedParticleState
from stonesoup.types.array import StateVectors

# Sample from the prior Gaussian distribution
states = multivariate_normal.rvs(np.array([0, 1, 0, 1]),
                                  np.diag([1., 1., 1., 1.]),
                                  size=number_particles)
covars = np.stack([np.eye(4) * 100 for i in range(number_particles)], axis=2) # (M, M, N)

# Create prior particle state.
prior = MarginalisedParticleState(
    state_vector=StateVectors(states.T),
    covariance=covars,
    weight=np.array([Probability(1/number_particles)]*number_particles),
                      timestamp=start_time-timedelta(seconds=1))

from stonesoup.types.hypothesis import SingleHypothesis
from stonesoup.types.track import Track

track = Track()
for measurement in measurements:
    prediction = predictor.predict(prior, timestamp=measurement.timestamp)
    hypothesis = SingleHypothesis(prediction, measurement)
    post = updater.update(hypothesis)
    track.append(post)
    prior = track[-1]
    print(f"track length ={len(track)} of {len(measurements)}")

# Implement Particle Smoother Class

In [ ]:
from stonesoup.smoother.particle import MarginalisedKalmanSmoother, ParticleSmoother, CarterKohnSmoother
particlesmoother=ParticleSmoother()
culled_track=particlesmoother.particle_paths(track=track)

RTSsmoother=MarginalisedKalmanSmoother()
RTS_track=RTSsmoother.smooth(track=track)

CKsmoother=CarterKohnSmoother()
CK_track=CKsmoother.smooth(track=track)

In [ ]:
calc_OSPA=False
plot_OSPA=False
uncertainty=True
particle=True
plot_particle_paths=True
show_1D=False
save_1D=False
show_2D=False
save_2D=False

## Testing Smoothing

In [ ]:
if calc_OSPA or plot_OSPA:
    tracking_filters = ["unsmoothed", 
                        "mean_descendant_smoothed", 
                        "mean_RTS_smoothed", 
                        "mean_CK_smoothed"                               ]

    from stonesoup.metricgenerator.ospametric import OSPAMetric

    ospa_generators = [OSPAMetric(c=40, p=1,
                                generator_name=f'{tracking_filter} OSPA metrics',
                                tracks_key=f'tracks_{tracking_filter}',
                                truths_key='truths'
                                )
                    for tracking_filter in tracking_filters]

    from stonesoup.metricgenerator.tracktotruthmetrics import SIAPMetrics
    from stonesoup.measures import Euclidean

    siap_generators = [SIAPMetrics(position_measure=Euclidean((0, 2)),
                                velocity_measure=Euclidean((1, 3)),
                                generator_name=f'{tracking_filter} SIAP metrics',
                                tracks_key=f'tracks_{tracking_filter}',
                                truths_key='truths'
                                )
                    for tracking_filter in tracking_filters]

    from stonesoup.metricgenerator.uncertaintymetric import SumofCovarianceNormsMetric

    uncertainty_generators = [
        SumofCovarianceNormsMetric(generator_name=f'{tracking_filter} OSPA metrics',
                                tracks_key=f'tracks_{tracking_filter}')
        for tracking_filter in tracking_filters]

    from stonesoup.dataassociator.tracktotrack import TrackToTruth
    from stonesoup.metricgenerator.manager import MultiManager

    associator = TrackToTruth(association_threshold=30)

    generators = ospa_generators + siap_generators + uncertainty_generators
    metric_manager = MultiManager(generators, associator=associator)

    metric_manager.add_data({'truths': [truth],
                            'tracks_unsmoothed': [track],
                            'tracks_mean_descendant_smoothed': [culled_track],
                            'tracks_mean_RTS_smoothed': [RTS_track],
                            'tracks_mean_CK_smoothed': [CK_track]
                            })  
    metrics = metric_manager.generate_metrics()

    from stonesoup.plotter import MetricPlotter

    # sum up distance error from ground truth over all timestamps
    for tracking_filter in tracking_filters:
        total = sum([metrics[f'{tracking_filter} OSPA metrics']['OSPA distances'].value[i].value
                    for i in range(0, len(metrics[f'{tracking_filter} OSPA metrics']['OSPA distances'].value))])
        print(f'OSPA total value for {tracking_filter} is {total:.3f}')
    
    if plot_OSPA:
        fig1 = MetricPlotter()
        fig1.plot_metrics(metrics, metric_names=['OSPA distances'])

### 1D  Plot

In [ ]:
from pathlib import Path
from stonesoup.plotter import AnimatedPlotterly, Plotterly,  Dimension
folder_path=rf"C:\Users\joesb\OneDrive\Documents\Cambridge\IIB\PROJECT- Implementation of N-G TAs in SS framework\CKPlots"

In [ ]:
axis_label_list=["x","dx_dt","y","dy_dt"]
particle_plotter_dict = {}
if save_1D or show_1D:
    for i,label in enumerate(axis_label_list):
        file_path = Path(folder_path + rf"\1D_plot_{label}_{num_steps}steps_{number_particles}p.html")
        file_path.parent.mkdir(parents=True, exist_ok=True)
        
        particle_plotter_dict[label]= Plotterly(autosize=False, width=1500,height=800, dimension=Dimension.ONE, axis_labels=[label])
        particle_plotter_dict[label].plot_ground_truths(truth, [i],mode="lines", line=dict(width=2))
        if label =="x" or label=="y":
            particle_plotter_dict[label].plot_measurements(measurements, [i],marker=dict(symbol="x",size=4))
            
        particle_plotter_dict[label].plot_tracks(track, [i],mode="lines",uncertainty=uncertainty,particle=particle,plot_particle_paths=plot_particle_paths, track_label="Filtered",line=dict(width=1))
        particle_plotter_dict[label].plot_tracks(culled_track, [i],uncertainty=uncertainty,particle=particle,mode="lines",plot_particle_paths=plot_particle_paths, track_label="\'Descendant\'",line=dict(width=1))
        particle_plotter_dict[label].plot_tracks(RTS_track, [i],uncertainty=uncertainty,particle=particle,mode="lines",plot_particle_paths=plot_particle_paths,track_label="RTS",line=dict(width=1))
        particle_plotter_dict[label].plot_tracks(CK_track,[i],uncertainty=uncertainty,particle=particle, mode="lines",plot_particle_paths=plot_particle_paths,track_label="CK",line=dict(width=1))

        particle_plotter_dict[label].fig.update_layout( 
            plot_bgcolor="white",  # Set background color to white
            xaxis=dict(
                showgrid=True,
                gridcolor="gray",      # Keep the grid
                title=dict(text="Time", font=dict(size=20)),  # Add large label
            ),
            yaxis=dict(
                showgrid=True,
                gridcolor="gray",      # Keep the grid
                title=dict(text=label, font=dict(size=20)),  # Add large label
            ),
            legend=dict(
                font=dict(size=15),       # Make the legend font larger
                # orientation='v',
                # xanchor="auto",         # Center the legend
                # yanchor="auto",           # Align the legend to the bottom of the plot
                bordercolor="Black",
                borderwidth=3,
                # y=+0.45,                   # Position it above the graph
                # x=0.6                    # Center it horizontally
            ),
        )
        if save_1D:
            particle_plotter_dict[label].fig.write_html(str(file_path))
        if show_1D:
            particle_plotter_dict[label].fig.show()

### 2D Plot 

In [ ]:
animated=False
animated_label_list= ["xy_position", "xy_velocity"]
animated_mappings =[[0,2],[1,3]]
animated_plotter_dict={}
if save_1D or show_1D:
    for i,label in enumerate(animated_label_list):
        file_path =Path(folder_path+ rf"\2D_interactive_plot_{label}_{num_steps}steps_{number_particles}p.html")
        file_path.parent.mkdir(parents=True, exist_ok=True)

        if label=="xy_velocity":
            axis_labels=["dx/dt","dy/dt"]
        else:
            axis_labels=["x","y"]
        if animated:
            animated_plotter_dict[label]= AnimatedPlotterly(timesteps, tail_length=1, width=1200, height=1200,
                                                        xaxis=dict(title=dict(text=f"<i>{axis_labels[0]}</i>")),
                                                        yaxis=dict(title=dict(text=f"<i>{axis_labels[1]}</i>")))
        else:
            animated_plotter_dict[label]= Plotterly(autosize=True, axis_labels=axis_labels)
        animated_plotter_dict[label].plot_ground_truths(truth, animated_mappings[i],line=dict(width=2))
        if label== "xy_position":
            animated_plotter_dict[label].plot_measurements(measurements, animated_mappings[i],marker=dict(symbol="x",size=5))
        animated_plotter_dict[label].plot_tracks(track, animated_mappings[i], particle=particle, uncertainty=uncertainty,plot_particle_paths=plot_particle_paths,mode="lines",track_label="Filtered",line=dict(width=2))
        animated_plotter_dict[label].plot_tracks(culled_track, animated_mappings[i],particle=particle, uncertainty=uncertainty,plot_particle_paths=plot_particle_paths,mode="lines",track_label="\'Descendant\'",line=dict(width=2))
        animated_plotter_dict[label].plot_tracks(RTS_track, animated_mappings[i],particle=particle, uncertainty=uncertainty,plot_particle_paths=plot_particle_paths,mode="lines",track_label="RTS",line=dict(width=2))
        animated_plotter_dict[label].plot_tracks(CK_track, animated_mappings[i],particle=particle, uncertainty=uncertainty,plot_particle_paths=plot_particle_paths,mode="lines",track_label="CK",line=dict(width=2))
        animated_plotter_dict[label].fig.update_layout( 
            plot_bgcolor="white",  # Set background color to white
            xaxis=dict(
                showgrid=True,
                gridcolor="gray",      # Keep the grid
                title=dict(text=f"{axis_labels[0]}", font=dict(size=20))
            ),
            yaxis=dict(
                showgrid=True,
                gridcolor="gray",      # Keep the grid
                title=dict(text=f"{axis_labels[1]}", font=dict(size=20))
            ),
            legend=dict(
                font=dict(size=15),       # Make the legend font larger
                # orientation='v',
                # xanchor="auto",         # Center the legend
                # yanchor="auto",           # Align the legend to the bottom of the plot
                bordercolor="Black",
                borderwidth=3,
                # y=+0.45,                   # Position it above the graph
                # x=0.6                    # Center it horizontally
            ),
        )
        if save_2D:
            animated_plotter_dict[label].fig.write_html(str(file_path))
        if show_2D:
            animated_plotter_dict[label].fig.show()
    

## References
[1] Lemke, Tatjana, and Simon J. Godsill, 'Inference for models with asymmetric α -stable noise processes', in Siem Jan Koopman, and Neil Shephard (eds), Unobserved Components and Time Series Econometrics (Oxford, 2015; online edn, Oxford Academic, 21 Jan. 2016)

[2] S. Godsill, M. Riabiz, and I. Kontoyiannis, “The L ́evy state space model,” in 2019 53rd Asilomar Conference on Signals, Systems, and Computers, 2019, pp. 487–494.
